In [8]:
import gensim
import gensim.downloader as api
from gensim.models import KeyedVectors
from scipy.stats import spearmanr

In [9]:
model: KeyedVectors = api.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [10]:
vocab_size = len(model.key_to_index)
vector_dim = model.vector_size
vocab_size, vector_dim

(3000000, 300)

In [11]:
test_words = ['car', 'beautiful', 'play', 'computer']
for word in test_words:
    similar = model.most_similar(word, topn=10)
    for word, score in similar:
        print(f"{word}: {score:.4f}")
    print('='*50)

vehicle: 0.7821
cars: 0.7424
SUV: 0.7161
minivan: 0.6907
truck: 0.6736
Car: 0.6678
Ford_Focus: 0.6673
Honda_Civic: 0.6627
Jeep: 0.6511
pickup_truck: 0.6441
gorgeous: 0.8353
lovely: 0.8107
stunningly_beautiful: 0.7329
breathtakingly_beautiful: 0.7231
wonderful: 0.6854
fabulous: 0.6700
loveliest: 0.6613
prettiest: 0.6595
beatiful: 0.6593
magnificent: 0.6591
playing: 0.7668
plays: 0.7108
played: 0.6962
game: 0.6501
toplay: 0.5970
Playing: 0.5813
games: 0.5265
paly: 0.5261
score: 0.5229
Play: 0.5014
computers: 0.7979
laptop: 0.6640
laptop_computer: 0.6549
Computer: 0.6473
com_puter: 0.6082
technician_Leonard_Luchko: 0.5663
mainframes_minicomputers: 0.5618
laptop_computers: 0.5585
PC: 0.5540
maker_Dell_DELL.O: 0.5519


In [13]:
similarity_pairs = [
    ("car", "car", 9),
    ("computer", "laptop", 8),
    ("sun", "moon", 6),
    ("cat", "dog", 7),
    ("joy", "happiness", 8),
    ("house", "building", 7),
    ("fast", "fast", 8),
    ("red", "scarlet", 9),
    ("good", "bad", 4)
]
cosine_similarities = []
human_scores = []
for w1, w2, score in similarity_pairs:
    cos_sim = model.similarity(w1, w2)
    cosine_similarities.append(cos_sim)
    human_scores.append(score)
    print(f"{w1} - {w2} : cos_sim = {cos_sim:.4f}, human_scores = {score}")
res = spearmanr(cosine_similarities, human_scores)
print(f"\nSpearman: {res.statistic:.4f}, p_value: {res.pvalue:.4f}")

car - car : cos_sim = 1.0000, human_scores = 9
computer - laptop : cos_sim = 0.6640, human_scores = 8
sun - moon : cos_sim = 0.4263, human_scores = 6
cat - dog : cos_sim = 0.7609, human_scores = 7
joy - happiness : cos_sim = 0.6183, human_scores = 8
house - building : cos_sim = 0.4379, human_scores = 7
fast - fast : cos_sim = 1.0000, human_scores = 8
red - scarlet : cos_sim = 0.5054, human_scores = 9
good - bad : cos_sim = 0.7190, human_scores = 4

Spearman: 0.2962, p_value: 0.4390


#### Conclusion(1.3):
- A ρ value of 0.2962 is very close to zero, indicating almost no monotonic relationship between the model’s cosine similarity scores and human similarity scores.
In other words, the embedding similarities are poorly aligned with human judgments for this small set of word pairs.
- p-value = 0.4390:
The p-value tests the null hypothesis that there is no correlation.
A p-value this high (much greater than 0.05) means we cannot reject the null hypothesis, reinforcing that the observed correlation is not statistically significant.
Some identical or very similar words like "car-car" and "red-scarlet" might have high cosine similarity, but other pairs like "good-bad" (antonyms) have unexpectedly high or low similarity that doesn’t match human judgment.
With only 9 word pairs, the dataset is very small, so the Spearman correlation is also sensitive to a few mismatched pairs.

In [16]:
result = model.most_similar(positive=['king', 'queen'],
                            negative=['king'])

for word, sim in result:
    print(f"{word:<15} {sim:.4f}")

queens          0.7399
princess        0.7071
monarch         0.6384
very_pampered_McElhatton 0.6357
Queen           0.6163
NYC_anglophiles_aflutter 0.6061
Queen_Consort   0.5924
princesses      0.5908
royal           0.5637
Makobo_Modjadji 0.5570


In [18]:
def test_identity_analogy(a, b):
    result = model.most_similar(
        positive=[a, b], negative=[a], topn=1)
    print(
        f"{a:} - {a:} + {b:} === {result[0][0]:} (sim: {result[0][1]:.4f})")


print("--- 3 reasonable analogy answers ---")
test_identity_analogy('king', 'queen')
test_identity_analogy('car', 'vehicle')
test_identity_analogy('man', 'doctor')

print("\n--- 2 mistakes versus the answer ---")
test_identity_analogy('eat', 'apple')
test_identity_analogy('increase', 'decrease')

--- 3 reasonable analogy answers ---
king - king + queen === queens (sim: 0.7399)
car - car + vehicle === SUV (sim: 0.7578)
man - man + doctor === physician (sim: 0.7806)

--- 2 mistakes versus the answer ---
eat - eat + apple === apples (sim: 0.7204)
increase - increase + decrease === decreases (sim: 0.8094)


### 3 Reasonable Examples (Word2Vec-Google-News-300)
- king – king + queen → queens
Reason: Plural form is semantically identical and highly similar in vector space.
- car – car + vehicle → SUV
Reason: SUV is a typical hyponym of vehicle, showing valid semantic similarity.
- man – man + doctor → physician
Reason: Physician is a close synonym, reflecting strong lexical similarity.

### 2 Wrong Examples
- eat – eat + apple → apples
Reason: Model only captures morphological (plural) form, not deeper semantic relations.
- increase – increase + decrease → decreases
Reason: Model prioritizes inflectional form over antonymy or conceptual meaning.